In [ ]:
import math
import random

In [ ]:
def ackley(vector):
  n=len(vector)
  a=20
  sum_sq=0
  sum_cos=0
  for i in vector:
    sum_sq += i**2
    sum_cos += math.cos(2*math.pi*i)
  termi1 = -a*math.exp(-0.2*math.sqrt(sum_sq/n))
  termi2 = -math.exp(sum_cos/n)
  energy = termi1+termi2+a+math.e
  return energy


In [ ]:
#this one has dependent terms inside summation
def rosenbrock(vector):
  n = len(vector)
  a=1
  b=100
  energy=0
  for i in range(n-1):
    termi1 = b*(vector[i+1]-vector[i]**2)**2
    termi2 = (a-vector[i])**2
    energy += termi1+termi2
  return energy

In [ ]:
def sphere(vector):
  #f(x) = sum(1 to n)(xi^2)
  energy = 0
  for i in vector:
    termi = i**2
    energy += termi
  return energy

In [ ]:
def rastrigin(vector):
  #f(x1, x2) = 20 + x1^2 + x2^2 -10(cos(2*pi*x1)+cos(2*pi*x2))
  energy=10*len(vector)
  for i in vector:
    termi = i**2 - 10*math.cos(2*math.pi*i)
    energy += termi
  return energy

In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter
import cupy as cp
print(f"GPU Detected: {cp.cuda.Device(0).attributes['MultiProcessorCount']} multiprocessors ready.")

In [ ]:
%%writefile kernels.cu

extern "C" {

  #define DIMS 30
  #define PI 3.1415926535f

  __global__ void rastriginKernel(const float *vector, float *energy, int threads){
      int t_idx = blockIdx.x * blockDim.x + threadIdx.x;
      if(t_idx<threads){
          float e = 10.0f*DIMS;
          for(int i=0;i<DIMS;i++){
              int flat_idx = (t_idx*DIMS)+i;
              float x = vector[flat_idx];
              float termi = (x*x)-10.0f*cosf(2.0f*PI*x);
              e+=termi;
          }
          energy[t_idx] = e;
      }
  }

  __global__ void sphereKernel(const float *vector, float *energy, int threads){
      int t_idx = blockIdx.x * blockDim.x + threadIdx.x;
      if(t_idx<threads){
          float e = 0.0f;
          for(int i=0;i<DIMS;i++){
              int flat_idx = (t_idx*DIMS)+i;
              float x = vector[flat_idx];

              e += (x*x);
          }
          energy[t_idx] = e;
      }
  }

    __global__ void ackleyKernel(const float *vector, float *energy, int threads){
        int t_idx = blockIdx.x * blockDim.x + threadIdx.x;
        if(t_idx<threads){
            float a =20.0f;
            float sq_sum=0.0f;
            float sum_cos=0.0f;
            float e = 0.0f;
            for(int i=0;i<DIMS;i++){
                int flat_idx = (t_idx*DIMS)+i;
                float x = vector[flat_idx];
                sq_sum += (x*x);
                sum_cos += cosf(2.0f*PI*x);
            }
            float termi1 = -a*expf(-0.2f*sqrtf(sq_sum/(float)DIMS));
            float termi2 = -expf(sum_cos/(float)DIMS);
            e = termi1 + termi2 + a + expf(1.0f);
            energy[t_idx] = e;
        }
    }
    __global__ void rosenbrockKernel(const float *vector, float *energy, int threads){
        int t_idx = blockIdx.x * blockDim.x + threadIdx.x;
        if(t_idx<threads){
            float a = 1.0f;
            float b = 100.0f;
            float e = 0.0f;
            for(int i=0;i<DIMS-1;i++){
                int flat_idx = (t_idx*DIMS)+i;
                float x_i = vector[flat_idx];
                float x_next = vector[flat_idx+1];

                float diff1 = x_next  -(x_i*x_i);
                float diff2 = a-x_i;
                float termi1 = b*(diff1*diff2);
                float termi2 = diff2*diff2;
                e += termi1+termi2;
            }
            energy[t_idx] = e;
        }
    }
    /*
    int main(){
        int num_threads = 100000;
        size_t vector_size = num_threads*DIMS*sizeof(float);
        size_t energy_size = num_threads*sizeof(float);

        float *h_vec = (float*)malloc(vector_size);
        float *h_eng = (float*)malloc(energy_size);

        for(int i =0;i<num_threads*DIMS;i++){
            h_vec[i] = 0.05f;
        }
        float *d_vec;
        cudaMalloc((void**)&d_vec, vector_size);
        float *d_eng;
        cudaMalloc((void**)&d_eng, energy_size);

        cudaMemcpy(d_vec, h_vec, vector_size, cudaMemcpyHostToDevice);
        int threads_per_block = 256;
        int block_per_grid = (num_threads + threads_per_block-1)/threads_per_block;

        rastriginKernel<<<block_per_grid, threads_per_block>>>(d_vec, d_eng, num_threads);
        cudaMemcpy(h_eng, d_eng, energy_size, cudaMemcpyDeviceToHost);
        std::cout<<"Result Verified. Thread 0 energy: "<<h_eng[0]<<std::endl;
        cudaFree(d_vec);
        cudaFree(d_eng);
        free(h_vec);
        free(h_eng);
        return 0;
    }
    */
}

In [ ]:
import cupy as cp
with open('kernels.cu', 'r') as f:
  stress_functions = f.read()
module = cp.RawModule(code=stress_functions)

kernels = {
    'ackley' : module.get_function('ackleyKernel'),
    'rosenbrock' : module.get_function('rosenbrockKernel'),
    'rastrigin' : module.get_function('rastriginKernel'),
    'sphere' : module.get_function('sphereKernel')
}

class SimulatedAnnealer:
  def __init__(self,current_vector, T, function_key, threads = 10000):
    self.gpu_func = kernels[function_key]
    self.threads=threads

    #10k clones on the GPU
    self.current_vector = cp.repeat(cp.array(current_vector, dtype=cp.float32)[cp.newaxis, :], threads, axis=0)

    self.energies = cp.zeros(threads, dtype=cp.float32)
    self._evaluate_gpu(self.current_vector)
    self.current_energy = self.energies.copy()

    self.best_pos_vector = cp.array(current_vector, dtype=cp.float32)
    self.best_energy = float(self.current_energy.min())
    self.T = T
    self.T_min = 0.000001
    self.alpha = 0.99999
    self.k_max = 5000000

  #connection to C++ code
  def _evaluate_gpu(self, vectors):
    t_pb = 256
    grid = (self.threads + t_pb-1)//t_pb
    self.gpu_func((grid,), (t_pb,),(vectors, self.energies, self.threads) )

  def SA(self):
    for i in range(0, self.k_max):

      if(self.T<=self.T_min):
        break

      initial_T = 500.0
      noise_radius = max((self.T / initial_T) * 2.5, 0.001)
      noise = cp.random.normal(0,noise_radius, (self.threads, 30)).astype(cp.float32)
      new_pos = cp.clip(self.current_vector+noise, -5.12, 5.12)

      self._evaluate_gpu(new_pos)
      new_energy = self.energies.copy()

      del_e = new_energy-self.current_energy
      p = cp.exp(-del_e/self.T)
      rng = cp.random.uniform(0,1, self.threads)

      accept = (del_e<0) | (rng<p)

      self.current_vector[accept] = new_pos[accept]
      self.current_energy[accept] = new_energy[accept]

      best_agent_idx = int(self.current_energy.argmin())
      current_min = float(self.current_energy[best_agent_idx])

      if current_min < self.best_energy:
        self.best_energy = current_min
        self.best_pos_vector = self.current_vector[best_agent_idx].copy()

      self.T *=self.alpha

      if i % 10000 == 0:
        print(f"Iteration {i} | T: {self.T:.5f} | Best Energy: {self.best_energy:.6f}")
    return self.best_energy, self.best_pos_vector.get().tolist()

In [ ]:
if __name__ == "__main__":
  start_pos = [0.5]*30
  A = SimulatedAnnealer(start_pos, T=10000.0, function_key='rastrigin')
  best_e, best_pos = A.SA()
  print("-"*30)
  print(f"Final Minimum Energy: {best_e:.8f}")
  print(f"Final Best Positions: {best_pos:.3f}")
